In [17]:
# check install for not common libraries 
# !pip install yfinance


In [18]:
# making sure API key is hidden 

In [19]:
import databento as db 
import pandas as pd 
import numpy as np 
import yfinance as yf
import warnings

warnings.filterwarnings('ignore')

from dotenv import load_dotenv



load_dotenv()
client = db.Historical()



In [20]:
def fetch_quantitative_data(ticker_symbol):
    print(f"📡 Fetching 15-minute data for {ticker_symbol}...")
    
    ticker_symbol = ticker_symbol.replace('.', '-')
    ticker = yf.Ticker(ticker_symbol)
    df = ticker.history(period="60d", interval="15m")
    
    if df.empty:
        raise ValueError(f"No data found for {ticker_symbol}. Check ticker symbol.")
        
    # --- THE FIX FOR yfinance's NEW UPDATE ---
    # If yfinance returned multi-level columns, this flattens them back to normal
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    # -----------------------------------------
        
    df.index = df.index.tz_convert('America/New_York')
    
    df['Typical_Price'] = (df['High'] + df['Low'] + df['Close']) / 3
    df['Price_Volume'] = df['Typical_Price'] * df['Volume']
    
    df['Date'] = df.index.date
    df['Cumulative_PV'] = df.groupby('Date')['Price_Volume'].cumsum()
    df['Cumulative_Volume'] = df.groupby('Date')['Volume'].cumsum()
    
    df['VWAP'] = df['Cumulative_PV'] / df['Cumulative_Volume']
    df['VWAP_Distance_Pct'] = round(((df['Close'] - df['VWAP']) / df['VWAP']) * 100, 3)
    
    df['Return'] = df['Close'].pct_change()
    rolling_mean = df['Return'].rolling(window=26).mean()
    rolling_std = df['Return'].rolling(window=26).std()
    df['Z_Score'] = round((df['Return'] - rolling_mean) / rolling_std, 3)
    
    rolling_vol_avg = df['Volume'].rolling(window=26).mean()
    df['Volume_Shock'] = round(df['Volume'] / (rolling_vol_avg + 1e-9), 2)
    
    df = df.dropna()
    cols_to_keep = ['Close', 'Volume', 'VWAP', 'VWAP_Distance_Pct', 'Z_Score', 'Volume_Shock']
    df_clean = df[cols_to_keep].rename(columns={'Close': 'Price'})
    
    print(f"✅ Successfully loaded {len(df_clean)} intervals.")
    return df_clean

# Test the newly fixed function!
quant_data = fetch_quantitative_data("BRK-B")
display(quant_data.tail(15))

📡 Fetching 15-minute data for BRK-B...
✅ Successfully loaded 1495 intervals.


,Price,Volume,VWAP,VWAP_Distance_Pct,Z_Score,Volume_Shock
Datetime,,,,,,
2026-04-20 12:15:00-04:00,472.750000,118563,474.161492,-0.298,0.766,0.71
2026-04-20 12:30:00-04:00,472.750092,124223,474.086602,-0.282,0.528,0.74
2026-04-20 12:45:00-04:00,472.954987,97001,474.039834,-0.229,0.967,0.58
2026-04-20 13:00:00-04:00,473.388611,113173,474.001797,-0.129,1.405,0.68
2026-04-20 13:15:00-04:00,473.440002,64601,473.989122,-0.116,0.487,0.39
2026-04-20 13:30:00-04:00,472.834991,130712,473.942082,-0.234,-1.022,0.79
2026-04-20 13:45:00-04:00,472.570007,82922,473.902642,-0.281,-0.210,0.50
2026-04-20 14:00:00-04:00,472.600006,125837,473.840697,-0.262,0.417,0.76
2026-04-20 14:15:00-04:00,472.934998,98257,473.808174,-0.184,1.147,0.59


In [24]:
# 1. Ask the user for the ticker and clean the input
# .strip() removes accidental spaces, .upper() ensures it matches API formatting
target_ticker = input("Enter a stock ticker (e.g., AAPL, TSLA, BRK-B): ").strip().upper()

# 2. Safety check to ensure the user actually typed something
if target_ticker:
    print(f"\n Initializing Quantitative Engine for {target_ticker}...\n")
    
    try:
        # 3. Feed the user's input directly into our data pipeline
        quant_data = fetch_quantitative_data(target_ticker)
        
        # 4. Display the resulting mathematical dataframe
        print(f"\n--- {target_ticker} Data Loaded ---")
        display(quant_data.head(25))
        
    except Exception as e:
        print(f" Error pulling data for {target_ticker}: {e}")
        print("Make sure you used a valid ticker symbol (use dashes for class shares, e.g., BRK-B).")
else:
    print(" No ticker entered. Please run the cell again.")


 Initializing Quantitative Engine for QQQ...

📡 Fetching 15-minute data for QQQ...
✅ Successfully loaded 1495 intervals.

--- QQQ Data Loaded ---


,Price,Volume,VWAP,VWAP_Distance_Pct,Z_Score,Volume_Shock
Datetime,,,,,,
2026-01-26 09:30:00-05:00,624.815002,2754300,624.002502,0.130,2.328,1.87
2026-01-26 09:45:00-05:00,625.294922,1327795,624.378454,0.147,0.414,0.90
2026-01-26 10:00:00-05:00,625.320007,1047939,624.529596,0.127,-0.150,0.72
2026-01-26 10:15:00-05:00,625.489990,4793386,625.085281,0.065,0.104,3.04
2026-01-26 10:30:00-05:00,625.330017,446322,625.107292,0.036,-0.304,0.29
2026-01-26 10:45:00-05:00,626.335022,834224,625.184062,0.184,1.344,0.54
2026-01-26 11:00:00-05:00,626.450012,799196,625.263468,0.190,0.076,0.52
2026-01-26 11:15:00-05:00,626.900024,795191,625.340711,0.249,0.494,0.52
2026-01-26 11:30:00-05:00,626.719971,793653,625.434484,0.206,-0.380,0.52


📡 Fetching 15-minute data for BRK-B...
✅ Successfully loaded 1495 intervals.


,Price,Volume,VWAP,VWAP_Distance_Pct,Z_Score,Volume_Shock
Datetime,,,,,,
2026-01-26 09:30:00-05:00,480.850006,453265,479.920634,0.194,2.445,2.77
2026-01-26 09:45:00-05:00,480.500000,139557,480.115876,0.080,-0.432,0.86
2026-01-26 10:00:00-05:00,480.950012,146843,480.245418,0.147,0.655,0.92
2026-01-26 10:15:00-05:00,481.825012,163416,480.476964,0.281,1.237,1.07
2026-01-26 10:30:00-05:00,481.184998,110217,480.565111,0.129,-1.107,0.74
2026-01-26 10:45:00-05:00,480.769989,100545,480.603315,0.035,-0.720,0.69
2026-01-26 11:00:00-05:00,480.449890,112447,480.598728,-0.031,-0.518,0.77
2026-01-26 11:15:00-05:00,479.945007,113337,480.556672,-0.127,-0.752,0.78
2026-01-26 11:30:00-05:00,480.709991,107554,480.541559,0.035,1.210,0.74
